<a href="https://colab.research.google.com/github/cout-angela/projectP_imaging/blob/main/Tentativo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Verifica della disponibilita' della GPU
import torch
print(f'CUDA disponibile: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CUDA disponibile: True
GPU: Tesla T4


In [ ]:
# ---------------------------------------------------------------------------
# Installazione delle dipendenze
# ---------------------------------------------------------------------------
# scikit-image : metriche PSNR/SSIM
# datasets     : HuggingFace, per il download del dataset
# numba        : richiesto da IPPy/solvers (usato nei solver TV)
# Nota: astra-toolbox NON viene installato perche' in IPPy serve solo
#       per CTProjector (tomografia), che non usiamo in questo progetto.
#       Installiamo IPPy dal sorgente e facciamo un patch del modulo
#       operators per evitare l'import di astra (vedi cella successiva).

!pip install scikit-image datasets numba tqdm --quiet

# Clone di IPPy nella directory di lavoro di Colab
import os
if not os.path.isdir('IPPy'):
    !git clone https://github.com/devangelista2/IPPy.git --quiet
    print('IPPy clonato.')
else:
    print('IPPy gia\' presente.')

IPPy clonato.


In [ ]:
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

# Add the parent directory of 'examples/' to sys.path
#sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

from IPPy import utilities, operators, solvers
from IPPy.utilities import load_image, save_image, normalize
from IPPy.utilities.metrics import PSNR, SSIM, RE

import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import scipy.ndimage

# Seed per riproducibilita'
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Utilizzo device: {DEVICE}')

# Parametri globali del progetto (NON modificare qui sotto)
IMG_SIZE    = 256
KERNEL_SIZE = 9
NOISE_LEVELS = [0.005, 0.01, 0.05, 0.1]

ModuleNotFoundError: No module named 'IPPy'

---
## 2. Dataset LSUN Church

Le istruzioni ufficiali per il download sono disponibili su: https://github.com/fyu/lsun

Il dataset e' in formato LMDB. La cella seguente mostra come scaricarlo e come costruire un Dataset PyTorch a partire da esso.

**Struttura attesa dopo il download:**
```
./data/
    church_outdoor_train_lmdb/
    church_outdoor_val_lmdb/
```

In [ ]:
# Download del dataset LSUN Church tramite lo script ufficiale
# Eseguire solo la prima volta; richiede ~3 GB di spazio.

!git clone https://github.com/fyu/lsun.git --quiet
!mkdir -p data

# TODO: scegliere la categoria corretta (church_outdoor) e avviare il download.
# Lo script download.py di fyu/lsun accetta come argomento la categoria.
# Documentarsi sulla sintassi esatta prima di eseguire.

# !python lsun/download.py -c church_outdoor -o data/

In [ ]:
import lmdb
import io

class LSUNChurchDataset(Dataset):
    """
    Dataset PyTorch per LSUN Church in formato LMDB.

    Parametri
    ----------
    lmdb_path : str
        Percorso alla cartella .lmdb del dataset.
    transform : callable, opzionale
        Trasformazioni da applicare alle immagini.
    max_samples : int, opzionale
        Se specificato, limita il dataset ai primi N campioni.
        Utile per sviluppo e debug.
    """

    def __init__(self, lmdb_path, transform=None, max_samples=None):
        self.lmdb_path = lmdb_path
        self.transform = transform

        # Apertura read-only del database LMDB
        env = lmdb.open(lmdb_path, readonly=True, lock=False, readahead=False)
        with env.begin() as txn:
            self.length = txn.stat()['entries']
        env.close()

        if max_samples is not None:
            self.length = min(self.length, max_samples)

        # Lista delle chiavi (lazy: aperta al primo accesso)
        self._env  = None
        self._keys = None

    def _open_env(self):
        if self._env is None:
            self._env = lmdb.open(
                self.lmdb_path, readonly=True, lock=False, readahead=False
            )
            with self._env.begin() as txn:
                cursor = txn.cursor()
                self._keys = [key for key, _ in cursor.iternext_dup(keys=True)]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        self._open_env()
        with self._env.begin() as txn:
            img_bytes = txn.get(self._keys[idx])
        img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img


# --- Preprocessing pipeline ---
# GIUSTIFICAZIONE SCELTE:
#   - Resize a 256x256: richiesto dalla traccia del progetto.
#   - CenterCrop prima del resize: rimuove bordi con artefatti JPEG frequenti
#     nel dataset LSUN, mantenendo il soggetto centrale.
#   - ToTensor: converte da PIL [0,255] a float32 [0.0, 1.0].
#   - Normalize con media 0.5 e std 0.5: porta i valori in [-1, 1].
#     ATTENZIONE: per le metriche PSNR/SSIM bisogna riportare in [0,1]
#     con la trasformazione inversa: x_01 = (x + 1) / 2.

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),                        # [0, 1]
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),    # [-1, 1]
])

def denormalize(tensor):
    """Riporta un tensore da [-1,1] a [0,1]."""
    return (tensor + 1.0) / 2.0


# --- Splitting del dataset ---
# TODO: definire le proporzioni dello split train/val/test.
#       Giustificare la scelta nella presentazione orale.
#       Verificare che non ci siano sovrapposizioni tra i set.
#
# Esempio (da completare con i path reali):
#
# TRAIN_LMDB = 'data/church_outdoor_train_lmdb'
# VAL_LMDB   = 'data/church_outdoor_val_lmdb'
#
# train_dataset = LSUNChurchDataset(TRAIN_LMDB, transform=preprocess, max_samples=5000)
# val_dataset   = LSUNChurchDataset(VAL_LMDB,   transform=preprocess, max_samples=500)
#
# Per il test set, estrarre un sottoinsieme separato dal train o usare
# l'intero val LMDB, documentando la scelta.

print('Dataset class definita. Completare i TODO sopra prima di procedere.')

In [ ]:
# TODO: caricare effettivamente il dataset una volta completato il download
# e verificare visivamente alcune immagini di esempio.

# Esempio di verifica visiva:
# loader = DataLoader(val_dataset, batch_size=4, shuffle=True)
# batch  = next(iter(loader))  # shape: (4, 3, 256, 256)
# batch_01 = denormalize(batch)  # riporta in [0,1] per la visualizzazione

# fig, axes = plt.subplots(1, 4, figsize=(14, 4))
# for i, ax in enumerate(axes):
#     img_np = batch_01[i].permute(1, 2, 0).numpy()
#     ax.imshow(img_np)
#     ax.axis('off')
#     ax.set_title(f'Campione {i+1}')
# plt.suptitle('Esempi dal dataset LSUN Church (preprocessed)')
# plt.tight_layout()
# plt.show()

print('TODO: completare questa cella dopo il download del dataset.')

---
## 2. Degradation - Motion Blur

In [ ]:

# Set device
device = utilities.get_device()
print(f"Device used: {device}.")

# Load GT image
x_true = load_image("../data/Mayo/test/C081/0.png")
print(f"Shape of the GT: {list(x_true.shape)}.")

# Create the blurring operator
kernel_size = 11
sigma = 1.5
K = operators.Blurring(
    img_shape=x_true.shape[-2:],
    kernel_type="gaussian",
    kernel_size=kernel_size,
    kernel_variance=sigma**2,
)

# Build test problem
noise_level = 0.01
y_delta = K(x_true)  + noise_level * torch.randn_like(x_true)
print(f"Shape of the measurements: {list(y_delta.shape)}.")

# Set up the solver: Tikhonov
lambda_tik=0.1
maxiter=100
tolf=tolx=1.e-7
solver = solvers.CGLS(K)


x_sol, info = solver(
    y_delta,
    x_true=x_true,
    starting_point=torch.zeros_like(x_true),
    lam=lambda_tik,
    maxiter=maxiter,
    tolf=tolf,
    tolx=tolx,
    verbose=True,
)
print(info.keys())

""" plt.figure()
plt.plot(info["SSIM"],'o')
plt.xlabel("iterations")
plt.ylabel("SSIM")
plt.show() """

# Compute metrics
psnr = PSNR(x_sol, x_true)
ssim = SSIM(x_sol, x_true)
re = RE(x_sol, x_true)
print(f"PSNR: {psnr:.2f} dB | SSIM: {ssim:.4f} | RE: {re:.4f}")

save_image(normalize(x_true), "gt_image.png")
save_image(normalize(y_delta), "blurred_image.png")
save_image(normalize(x_sol), "deblurred_image_tik.png")

# Set up the solver
lambda_tv = 1e-2
max_iters = 100
solver = solvers.ChambollePockTpVUnconstrained(K)

# Run the solver
x_sol, info = solver(
    y_delta,
    x_true=x_true,
    starting_point=torch.ones_like(x_true),
    lmbda=lambda_tv,
    max_iters=max_iters,
    p=0.5,
    verbose=True,
)

# Compute metrics
psnr = PSNR(x_sol, x_true)
ssim = SSIM(x_sol, x_true)
re = RE(x_sol, x_true)
print(f"PSNR: {psnr:.2f} dB | SSIM: {ssim:.4f} | RE: {re:.4f}")

# Save the results
save_image(normalize(x_sol), "deblurred_image_tpv.png")